In [1]:
# Construye y resuelve el modelo
import pyomo.environ as pe
import pyomo.opt as po

In [2]:
model = pe.ConcreteModel()

### Conjuntos


In [3]:
courses = ['A', 'B', 'C', 'D']
study_days = [1, 2, 3, 4]

model.courses = pe.Set(initialize=courses)
model.study_days = pe.Set(initialize=study_days)

### Parámetros


In [4]:
grade_dict = {
    ('A', 1): 3, ('A', 2): 5, ('A', 3): 6, ('A', 4): 7,
    ('B', 1): 4, ('B', 2): 5, ('B', 3): 6, ('B', 4): 9,
    ('C', 1): 2, ('C', 2): 5, ('C', 3): 7, ('C', 4): 8,
    ('D', 1): 5, ('D', 2): 6, ('D', 3): 7, ('D', 4): 9
}

# Un examen se aprueba con una nota igual o superior a 5
pass_dict = {key: int(grade >= 5) for key, grade in grade_dict.items()}

model.grade = pe.Param(model.courses, model.study_days, initialize=grade_dict)
model.passed = pe.Param(model.courses, model.study_days, initialize=pass_dict)

### Variables


In [5]:
# x[c, d] = 1 si se dedican d días a la asignatura c
model.x = pe.Var(model.courses, model.study_days, domain=pe.Binary)

### Función objetivo


#### Apartado a: maximizar el número de exámenes aprobados

Cada asignatura obtiene exactamente una nota según los días que se le asignen.


In [6]:
def obj_rule(model):
    return sum(
        model.passed[c, d] * model.x[c, d]
        for c in model.courses
        for d in model.study_days
    )

model.obj = pe.Objective(rule=obj_rule, sense=pe.maximize)

### Restricciones


In [7]:
def choose_one_duration(model, c):
    # Se elige una duración de entre 1 y 4 días para cada asignatura
    return sum(model.x[c, d] for d in model.study_days) == 1

model.choose_duration = pe.Constraint(model.courses, rule=choose_one_duration)

In [8]:
def use_all_days(model):
    # La suma de días dedicados a las cuatro asignaturas es siete
    return sum(d * model.x[c, d] for c in model.courses for d in model.study_days) == 7

model.total_days = pe.Constraint(rule=use_all_days)

## Resolución con Gurobi


In [9]:
solver = po.SolverFactory("gurobi_direct")
results_a = solver.solve(model, tee=True)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 7840HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 5 rows, 16 columns and 32 nonzeros (Max)
Model fingerprint: 0xf733ebcd
Model has 13 linear objective coefficients
Variable types: 0 continuous, 16 integer (16 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+00]

Presolve removed 0 rows and 7 columns
Presolve time: 0.00s
Presolved: 5 rows, 9 columns, 21 nonzeros
Variable types: 0 continuous, 9 integer (9 binary)
Found heuristic solution: objective 3.0000000
Found heuristic solution: objective 4.0000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 16 (of 16 available processors)

Solution count 2: 4 3 



In [10]:
def show_results(model, part):
    total_grade = 0
    passed_exams = 0

    print(f"\n{part}")
    for c in model.courses:
        selected_days = next(
            d for d in model.study_days if pe.value(model.x[c, d]) > 0.5
        )
        grade = pe.value(model.grade[c, selected_days])
        passed = grade >= 5
        total_grade += grade
        passed_exams += int(passed)
        status = "Aprobado" if passed else "Suspenso"
        print(f"{c}: {selected_days} día(s), nota {grade} ({status})")

    average_grade = total_grade / len(model.courses)
    print(f"Exámenes aprobados: {passed_exams}/{len(model.courses)}")
    print(f"Nota media: {average_grade:.2f}")

show_results(model, "Apartado a: máximo número de aprobados")


Apartado a: máximo número de aprobados
A: 2 día(s), nota 5 (Aprobado)
B: 2 día(s), nota 5 (Aprobado)
C: 2 día(s), nota 5 (Aprobado)
D: 1 día(s), nota 5 (Aprobado)
Exámenes aprobados: 4/4
Nota media: 5.00


#### Apartado b: maximizar la nota media

Como hay cuatro exámenes, maximizar la media equivale a maximizar la suma de las cuatro notas.


In [11]:
model.obj.deactivate()

def grade_obj_rule(model):
    return sum(
        model.grade[c, d] * model.x[c, d]
        for c in model.courses
        for d in model.study_days
    )

model.grade_obj = pe.Objective(rule=grade_obj_rule, sense=pe.maximize)

In [12]:
results_b = solver.solve(model, tee=True)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 7840HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 5 rows, 16 columns and 32 nonzeros (Max)
Model fingerprint: 0x5a4dd1d5
Model has 16 linear objective coefficients
Variable types: 0 continuous, 16 integer (16 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [2e+00, 9e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+00]

Presolve time: 0.00s
Presolved: 5 rows, 16 columns, 28 nonzeros
Variable types: 0 continuous, 16 integer (16 binary)
Found heuristic solution: objective 20.0000000
Found heuristic solution: objective 21.0000000

Root relaxation: cutoff, 5 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | I

In [13]:
show_results(model, "Apartado b: máxima nota media")


Apartado b: máxima nota media
A: 2 día(s), nota 5 (Aprobado)
B: 1 día(s), nota 4 (Suspenso)
C: 3 día(s), nota 7 (Aprobado)
D: 1 día(s), nota 5 (Aprobado)
Exámenes aprobados: 3/4
Nota media: 5.25
